In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.ToTensor()
train_ds = datasets.MNIST(root='../data', train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST(root='../data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=256)
print(f"训练: {len(train_ds)}, 测试: {len(test_ds)}")

训练: 60000, 测试: 10000


In [2]:
class RNN(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        # 输入28维(=一行的28个像素), 隐藏状态128维
        self.lstm = nn.LSTM(input_size=28, hidden_size=hidden, batch_first=True)
        self.fc   = nn.Linear(hidden, 10)

    def forward(self, x):
        # x: [B,1,28,28] -> 挤掉通道维 -> [B,28,28]
        x = x.squeeze(1)                  # 变成 (批次, 28个时间步, 每步28维)
        out, (h, c) = self.lstm(x)        # out: [B, 28, 128]
        return self.fc(out[:, -1, :])     # 只取最后时间步(读完整张图)的输出

model = RNN()
print(model)

RNN(
  (lstm): LSTM(28, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=10, bias=True)
)


In [3]:
images, labels = next(iter(train_loader))
print("原始图像:", images.shape)        # [64,1,28,28]
x = images.squeeze(1)
print("看作序列:", x.shape)             # [64,28,28] = 64样本, 28时间步, 每步28特征
out, (h, c) = model.lstm(x)
print("LSTM输出 out:", out.shape)       # [64,28,128]
print("隐藏状态 h:", h.shape, " 记忆单元 c:", c.shape)  # 都是 [1,64,128]

原始图像: torch.Size([64, 1, 28, 28])
看作序列: torch.Size([64, 28, 28])
LSTM输出 out: torch.Size([64, 28, 128])
隐藏状态 h: torch.Size([1, 64, 128])  记忆单元 c: torch.Size([1, 64, 128])


In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_one_epoch():
    model.train()
    total, correct, loss_sum = 0, 0, 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()          # 这里触发的就是 BPTT
        optimizer.step()
        loss_sum += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return loss_sum/len(train_loader), correct/total

for epoch in range(5):
    loss, acc = train_one_epoch()
    print(f"Epoch {epoch+1}: loss={loss:.4f}, 训练准确率={acc*100:.2f}%")

Epoch 1: loss=0.4768, 训练准确率=84.84%
Epoch 2: loss=0.1528, 训练准确率=95.41%
Epoch 3: loss=0.0986, 训练准确率=97.02%
Epoch 4: loss=0.0778, 训练准确率=97.65%
Epoch 5: loss=0.0656, 训练准确率=97.98%


In [5]:
def evaluate():
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return correct/total

print(f"测试集准确率: {evaluate()*100:.2f}%")

测试集准确率: 98.17%
